# Stage 2: Multi-agent Supervisor (LangGraph built in supervisor)


In [1]:
# %pip -q install langchain langchain-openai langgraph langgraph-supervisor python-dotenv

## 1) Imports

In [2]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph_swarm import create_swarm, create_handoff_tool
from typing import Any, Dict, Optional
from langchain_core.tools import tool
import os
from dotenv import load_dotenv

## 2) Load environment variables - please read instructions carefully

In [3]:
# if you are running in local, uncomment below line. also make sure you shall have a .env file
load_dotenv()

True

In [4]:
# if you are running in google colab, uncomment below line. and replace "Your_API_Key" with your own openAI API key
#os.environ["OPENAI_API_KEY"] = "Your_API_Key"

## 3) Helper: pretty_print()

In [5]:
def pretty_print(response):
    for m in response["messages"]:
        print("\n---", m.type, "---")
        print(m)

    last_msg = response["messages"][-1]

    if isinstance(last_msg.content, list):
        text = "".join(
            block["text"]
            for block in last_msg.content
            if block.get("type") == "text"
        )
    else:
        text = last_msg.content
    print(text)

## 4) Local tool: estimate_trip_cost

In [6]:

def estimate_trip_cost(
    destination: str,
    days: int,
    travelers: int,
    comfort: str = "mid",
) -> Dict[str, Any]:
    """
    Estimate a rough trip budget (SGD) using simple heuristics.
    comfort: budget | mid | premium
    Returns a breakdown and total estimate in SGD.
    """
    if days <= 0 or travelers <= 0:
        raise ValueError("days and travelers must be > 0")

    comfort = comfort.lower().strip()
    if comfort not in {"budget", "mid", "premium"}:
        raise ValueError("comfort must be one of: budget, mid, premium")

    # Very rough per-person-per-day estimates (SGD) excluding flights
    lodging_per_person_per_day = {"budget": 60, "mid": 140, "premium": 300}[comfort]
    food_per_person_per_day = {"budget": 30, "mid": 60, "premium": 120}[comfort]
    local_transport_per_person_per_day = {"budget": 10, "mid": 20, "premium": 50}[comfort]
    activities_per_person_per_day = {"budget": 20, "mid": 50, "premium": 120}[comfort]

    lodging = lodging_per_person_per_day * travelers * days
    food = food_per_person_per_day * travelers * days
    transport = local_transport_per_person_per_day * travelers * days
    activities = activities_per_person_per_day * travelers * days

    subtotal = lodging + food + transport + activities
    contingency = round(subtotal * 0.12)  # 12% buffer
    total = subtotal + contingency

    return {
        "destination": destination,
        "days": days,
        "travelers": travelers,
        "comfort": comfort,
        "currency": "SGD",
        "breakdown": {
            "lodging": lodging,
            "food": food,
            "local_transport": transport,
            "activities": activities,
            "contingency": contingency,
        },
        "total_estimate": total,
        "note": "Heuristic estimate excludes international flights/insurance/visa fees.",
    }



In [7]:
transfer_to_research_agent = create_handoff_tool(
    agent_name="research_agent",
    description="Transfer user to the research_agent.",
)

transfer_to_itinerary_agent = create_handoff_tool(
    agent_name="itinerary_agent",
    description="Transfer user to the itinerary_agent.",
)

transfer_to_cost_agent = create_handoff_tool(
    agent_name="cost_agent",
    description="Transfer user to the cost_agent.",
)

## 5) Create specialist agents

In [8]:
RESEARCH_SYSTEM = """You are ResearchAgent.
Your ONLY job: gather factual, practical travel info that will be useful for trip planning.
Rules:
- Use available tools when needed.
- if you're being asked for a full day-by-day plan + cost, that's not your job — redirect, don't answer directly.
- Output ONLY 5 bullets max.
- Each bullet: destination + why + travel time + one practical note.
"""
tools = [transfer_to_cost_agent, transfer_to_itinerary_agent]
web_tool = {"type": "web_search_preview"}
research_agent = create_agent(
    model=ChatOpenAI(model="gpt-4.1-mini", temperature=0.2, use_responses_api=True).bind(parallel_tool_calls=False),
    tools=[web_tool]+tools,
    system_prompt=RESEARCH_SYSTEM,
    name="research_agent"
)

In [9]:
COST_SYSTEM = """You are CostAgent.
Your ONLY job: compute total cost.
Rules:
- Never invent numbers, use tools avaiable.
- If destination/days/travelers/comfort are missing, use tools to gather them (max 2 questions) with available tools.
- If user says 'add an additional one-day trip', treat it as +1 day ONLY IF baseline days is known.
- Return: total cost + short assumptions.
"""

cost_agent = create_agent(
    model=ChatOpenAI(model="gpt-4.1-mini", temperature=0.1, use_responses_api=True).bind(parallel_tool_calls=False),
    tools=[estimate_trip_cost,transfer_to_research_agent, transfer_to_itinerary_agent],
    system_prompt=COST_SYSTEM,
    name="cost_agent",
)

In [10]:
ITINERARY_SYSTEM = """You are ItineraryAgent.
Your job: produce the final itinerary and incorporate research notes + cost breakdown.
Rules:
- Do NOT invent numeric costs; only use cost_breakdown if present.
- If cost_breakdown missing, use tool to gather required info (max 2 questions).
Output format:
1) Day-by-day plan (brief)
2) Total cost (SGD) + assumptions
"""

itinerary_agent = create_agent(
    model=ChatOpenAI(model="gpt-4.1-mini", temperature=0.4, use_responses_api=True).bind(parallel_tool_calls=False),
    tools=[transfer_to_cost_agent,transfer_to_research_agent],
    system_prompt=ITINERARY_SYSTEM,
    name="itinerary_agent",
)

## 6) Create swarm workflow

In [11]:
workflow = create_swarm(
    agents=[cost_agent,research_agent, itinerary_agent],
    default_active_agent="research_agent"
)

## 7) Demo

In [15]:
# Compile and run
app = workflow.compile()
user_prompt = "Plan a 2-day Tokyo trip for 2 adults, mid comfort. We like food and anime. Avoid packed schedules. give day by day plan and total cost."

result = app.invoke({"messages": [{"role": "user", "content": user_prompt}]})
pretty_print(result)

BadRequestError: Error code: 400 - {'error': {'message': 'No tool output found for function call call_1WfD1Ed6dCsvgynBgrBxar3E.', 'type': 'invalid_request_error', 'param': 'input', 'code': None}}